## Device

In [6]:
shared_folder_Path = "/media/marina/01DB47DC6EBB7DC0/ThreeD/shared"[:-6]
path = '/media/marina/01DB47DC6EBB7DC0/0_Data/conp-dataset/projects/calgary-campinas/CC359/Raw-data/Single-channel/IM_MoCo/'
path_model= '/media/marina/01DB47DC6EBB7DC0/ThreeD/IDEA_2/zFINAL_ImMoCoArtifact_CC359data/2_stacked/'
import sys,os, glob, torch
import pickle
sys.path.insert(1,shared_folder_Path)
try:
    os.mkdir('')
except:
    pass

paths_train = glob.glob(path + "/train/*")
paths_test  = glob.glob(path + "/test/*")
paths_test_low = [p for p in paths_test if ('motion_5' in p)]
paths_test_mid = [p for p in paths_test if ('motion_15' in p)]
paths_test_hgh = [p for p in paths_test if ('motion_25' in p)]
len(paths_train), len(paths_test_low), len(paths_test_hgh),len(paths_test)

(3, 400, 400, 1200)

## Import

In [7]:
%load_ext autoreload
%autoreload 2

from torcheval.metrics import PeakSignalNoiseRatio
import torch
from torch.optim import lr_scheduler
from torch.autograd import Variable
import numpy as np
import random
from tqdm import tqdm
import nibabel as nib
from shared.plot import plot_2_Imgs, plot_3_Imgs, plot_4_Imgs, plot_Img
from torch.utils.data import Dataset, DataLoader
from shared.ssim import SSIM
ssim_calc = SSIM()
from torchvision import transforms
import torch.nn.functional as F
from scipy import ndimage
sys.path.append('src/')


def get_n_params( model):
    pp=0
    for p in list(model.parameters()):
        nn=1
        for s in list(p.size()):
            nn = nn*s
        pp += nn

    return pp


# Deterministic for spectral norm
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
def init_seeds(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    random.seed(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(0)
    # no change in alg
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.use_deterministic_algorithms(True)
init_seeds(42)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Data

In [8]:
import torch
from torch.fft import fftn, fftshift, ifftn, ifftshift
import torchvision
import torch.nn.functional as F

vflip = lambda a: torchvision.transforms.functional.vflip(a)
hflip = lambda a: torchvision.transforms.functional.hflip(a)


def pad(arr,pad_to):

    arr = F.pad( arr, ((pad_to-arr.shape[1])//2, (pad_to-arr.shape[1])//2,
                       (pad_to-arr.shape[0])//2, (pad_to-arr.shape[0])//2), mode='constant', value=0)

    return arr


def real_2_complex_shape(arr):
    if(len(arr.shape)==4 and arr.shape[1]==2):
        k_out2 = arr[:,0:1,:,:] + 1j* arr[:,1:2,:,:]
    if(len(arr.shape)==3 and arr.shape[0]==2):
        k_out2 = arr[0:1,:,:] + 1j* arr[1:2,:,:]

    return k_out2

def print_mini_maxi(arr):
    print(arr.min(), arr.max())
    
def FFT(x):
    return fftshift(fftn(ifftshift(x, dim=(-2, -1)), dim=(-2, -1)), dim=(-2, -1))


def IFFT(x):
    return ifftshift(ifftn(fftshift(x, dim=(-2, -1)), dim=(-2, -1)), dim=(-2, -1))


def norm_0_1(arr):
    arr = ((arr - arr.min()) / (arr.max()-arr.min()))
    return arr
def norm_1_1(arr):
    arr = ((arr - arr.min()) / (arr.max()-arr.min()))*2-1
    return arr


def normImg(x: torch.Tensor):

    # group norm
    c, h, w = x.shape
    x = x.reshape(1, c // 1 * h * w)
  
    mean = x.mean(dim=2).view(1, 1, 1)
    std = x.std(dim=2).view(1, 1, 1)
    x = x.view(c, h, w)
  
  
    return (x - mean) / std, mean, std


def convert_polar_to_cylindrical( magnitude, phase):

    real = magnitude * torch.cos(phase)
    imag = magnitude * torch.sin(phase)
    return real, imag


def convert_cylindrical_to_polar(real,imag):

    mag = (real ** 2 + imag ** 2) ** (0.5)
    phase = torch.atan2(imag, real)
    phase[phase.ne(phase)] = 0.0  # remove NANs if any
    return mag, phase

def normalized_complex_mag(arr):

    mag, phase = convert_cylindrical_to_polar(arr.real,arr.imag)
    normalized_magnitude = (mag - torch.mean(mag)) / (torch.std(mag))
    real,imag = convert_polar_to_cylindrical(normalized_magnitude,phase)
    normed = real+1j*imag 

    return normed


In [9]:
def preprocess(kspace, k_space_artifact,Main_Artifact_path,mask):

    # IFFT
    image = IFFT(kspace).abs()
    image_artifact = IFFT(k_space_artifact).abs()
    
    # prior
    num_diff = 6
    num =  int(Main_Artifact_path[ Main_Artifact_path.find("slice")+num_diff:Main_Artifact_path.find("_motion")])
    path_aft = Main_Artifact_path.replace('slice_' + str(num), 'slice_' +  str(num+1))
    path_bef = Main_Artifact_path.replace('slice_' + str(num),'slice_' + str(num-1))
    
    try:
        _,K_bef_art,K_bef_mask = torch.load(path_bef, weights_only=True)
        bef_art = IFFT(K_bef_art).abs()
    except:
        K_bef_art = k_space_artifact
        K_bef_mask = mask
        bef_art = image_artifact

    try:
        _,K_aft_art,K_aft_mask = torch.load(path_aft, weights_only=True)
        aft_art = IFFT(K_aft_art).abs()
    except:
        K_aft_mask = mask
        K_aft_art = k_space_artifact
        aft_art = image_artifact

    # Norm img
    image = norm_0_1(image)
    image_artifact = norm_0_1(image_artifact)
    bef_art = norm_0_1(bef_art)
    aft_art = norm_0_1(aft_art)    

    # View
    kspace = torch.view_as_real(kspace)
    kspace = torch.moveaxis(kspace,-1,0)
    k_space_artifact = torch.view_as_real(k_space_artifact)
    k_space_artifact = torch.moveaxis(k_space_artifact,-1,0)
    K_bef_art = torch.view_as_real(K_bef_art)
    K_bef_art = torch.moveaxis(K_bef_art,-1,0)
    K_aft_art = torch.view_as_real(K_aft_art)
    K_aft_art = torch.moveaxis(K_aft_art,-1,0)


    return  {"Arr":image[None,...],
                "Art":image_artifact[None,...],
                
                "K_bef_art":K_bef_art,
                "bef_art":bef_art[None,...],
                
                "K_aft_art":K_aft_art,
                "aft_art":aft_art[None,...],

                "K_Arr":kspace,
                "K_Art":k_space_artifact,
                
                'path':Main_Artifact_path,
                'mask':abs(mask)[None],
                'mask_bef':abs(K_bef_mask)[None],
                'mask_aft':abs(K_aft_mask)[None],
            }

class MRIDataset(Dataset):
    def __init__(self, imgs_paths):
        self.imgs_paths = imgs_paths
    def __len__(self):
        return len(self.imgs_paths)
    def __getitem__(self, index):

        Main_Artifact_path = self.imgs_paths[index]
        kspace,k_space_artifact,mask = torch.load(Main_Artifact_path, weights_only=False)
        return  preprocess(kspace,k_space_artifact,Main_Artifact_path, mask)

In [10]:
#dataset
dataset_train = MRIDataset(paths_train)
dataset_val = MRIDataset(paths_test)
print(len(dataset_train), len(dataset_val))

#dataloaders
batch_size = 1
dataloaders = dict()
dataloaders['train'] = DataLoader(dataset_train, batch_size=batch_size,
                                   shuffle=True, pin_memory=True)
dataloaders['val'] = DataLoader(dataset_val, batch_size=batch_size,
                                shuffle=False, pin_memory=True)
sample = dataset_val[random.randint(0,len(dataset_val))]

device = 'cuda'    
with torch.no_grad():
    print(sample['path'])
    Arr = sample['Arr'][None].to(device)
    Art = sample['Art'][None].to(device)
    
    K_Art = sample['K_Art'].to(device)
    K_Arr = sample['K_Arr'].to(device)
    K_Arr = torch.view_as_complex(torch.moveaxis(K_Arr,0,-1))

3 1200
/media/marina/01DB47DC6EBB7DC0/0_Data/conp-dataset/projects/calgary-campinas/CC359/Raw-data/Single-channel/IM_MoCo//test/e14258s3_P76800.7._slice_78_motion_25.pt


## Model

In [11]:
from src.models.kld_net import get_unet
net = get_unet(in_chans=2, out_chans=1, chans=32, num_pool_layers=4, drop_prob=0.0).cuda()
net = torch.load('/media/marina/01DB47DC6EBB7DC0/ThreeD/IDEA_2/zFINAL_ImMoCoArtifact_CC359data/4_IM_MoCo/MICCAI24_IMMoCo-main copy/29model.pt')

/tmp/ipykernel_31177/4275484713.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  net = torch.load('/media/marina/01DB47DC6EBB7DC0/ThreeD/IDEA_2/zFINAL_ImMoCoArtifact_CC35

In [ ]:
from torcheval.metrics import PeakSignalNoiseRatio
metric = PeakSignalNoiseRatio()
from shared.ssim import SSIM
SSIM_criterion = SSIM().cuda()
from shared.CreateRoffetBlur import CreteRoffetBlur
b  = CreteRoffetBlur(2)
from shared.mutualInfo import mutual_information

from src.models import immoco
from src.utils.motion_utils import extract_movement_groups

In [10]:
pbar_test = tqdm(enumerate(dataloaders['val']), total=len(dataloaders['val']), position=0, leave=True)
for (i, datatest) in pbar_test:
    ssim_all = [] ; psnr_bef=[] ; ssim_all_bef = []
    psnr=[] ; BLUR = [] ; BLUR_BEF = []
    MI = [] ; MI_BEF = []   

    # KLD    
    torch.use_deterministic_algorithms(True)
    Art = Variable(datatest['Art']).to(device)
    Arr = Variable(datatest['Arr']).to(device)
    K_Art = Variable(datatest['K_Art']).to(device)
    path_ = (datatest['path'])[0]
    
    mask = net( K_Art)
    mask = (mask.sigmoid()   > 0.5  ).float()
    masks = extract_movement_groups(
        mask.squeeze().sum(0).div(mask.squeeze().shape[0]) > 0.2, make_list=True
    )
    
    # test immoco
    torch.use_deterministic_algorithms(False)
    K_Art = real_2_complex_shape(K_Art)[0,0]
    immoco_image, forward_kspace = immoco.imcoco_motion_correction(
        K_Art, masks, learning_rate=1e-2, lambda_ge=1e-2, debug=False)


    # Calc
    outputs_test = norm_0_1(abs(immoco_image))[None][None] 
    
    ssim_all .append(  float(SSIM_criterion(outputs_test, Arr).cpu().detach()))
    ssim_all_bef .append( float(SSIM_criterion(Art, Arr).cpu().detach()))

    metric = PeakSignalNoiseRatio()
    metric.update(outputs_test, Arr)
    paft = float(metric.compute())
    psnr.append( paft)

    metric = PeakSignalNoiseRatio()
    metric.update(Art, Arr)
    pbef = float(metric.compute())
    psnr_bef.append( pbef)

    BLUR .append( torch.sum(b(outputs_test)).cpu().detach())
    BLUR_BEF  .append( torch.sum(b(Art)).cpu().detach())


    MI .append( mutual_information(Arr,outputs_test))
    MI_BEF .append( mutual_information(Arr,Art))

    # save
    p = '/media/marina/01DB47DC6EBB7DC0/ThreeD/IDEA_2/zFINAL_ImMoCoArtifact_CC359data/4_IM_MoCo/MICCAI24_IMMoCo-main copy/data/'
    with open(p+ 'ssim_all.txt', 'a') as f:
        f.write( str(float(SSIM_criterion(outputs_test, Arr).cpu().detach())) + "\n" )
    with open(p+ 'ssim_all_bef.txt', 'a') as f:
        f.write( str(float(SSIM_criterion(Art, Arr).cpu().detach())) + "\n" )

    with open(p+ 'psnr_all.txt', 'a') as f:
        f.write( str(paft) + "\n" )
    with open(p+ 'psnr_all_bef.txt', 'a') as f:
        f.write( str(pbef) + "\n" )

    with open(p+ 'blur_all.txt', 'a') as f:
        f.write( str(float(torch.sum(b(outputs_test)).cpu().detach()))  + "\n" )
    with open(p+ 'blur_all_bef.txt', 'a') as f:
        f.write( str(float(torch.sum(b(Art)).cpu().detach())) + "\n" )
    
    with open(p+ 'MI_all.txt', 'a') as f:
        f.write( str(float(mutual_information(Arr,outputs_test)))  + "\n" )
    with open(p+ 'MI_all_bef.txt', 'a') as f:
        f.write( str(float(mutual_information(Arr,Art))) + "\n" )
    
    with open(p+ 'path.txt', 'a') as f:
        f.write( path_ + "\n" )

 84%|████████▎ | 1004/1200 [14:00:47<2:37:21, 48.17s/it]tiny-cuda-nn warning: FullyFusedMLP is not supported for the selected architecture 61. Falling back to CutlassMLP. For maximum performance, raise the target GPU architecture to 75+.
tiny-cuda-nn warning: FullyFusedMLP is not supported for the selected architecture 61. Falling back to CutlassMLP. For maximum performance, raise the target GPU architecture to 75+.
 84%|████████▍ | 1005/1200 [14:02:06<3:06:27, 57.37s/it]tiny-cuda-nn warning: FullyFusedMLP is not supported for the selected architecture 61. Falling back to CutlassMLP. For maximum performance, raise the target GPU architecture to 75+.
tiny-cuda-nn warning: FullyFusedMLP is not supported for the selected architecture 61. Falling back to CutlassMLP. For maximum performance, raise the target GPU architecture to 75+.
 84%|████████▍ | 1006/1200 [14:02:27<2:30:20, 46.50s/it]tiny-cuda-nn warning: FullyFusedMLP is not supported for the selected architecture 61. Falling back to C

In [11]:
print("ssim aft std :",np.std(ssim_all), "ssim mean :",np.mean(ssim_all))
print("ssim bef std :",np.std(ssim_all_bef), "ssim mean :",np.mean(ssim_all_bef))

print("psnr aft std :",np.std(psnr), "psnr mean :",np.mean(psnr))
print("psnr bef std :",np.std(psnr_bef), "psnr mean :",np.mean(psnr_bef))

print("Blur aft std :",np.std(BLUR), "Blur mean :",np.mean(BLUR))
print("Blur bef std :",np.std(BLUR_BEF), "Blur mean :",np.mean(BLUR_BEF))

print("MI aft std :",np.std(MI), "MI mean :",np.mean(MI))
print("MI bef std :",np.std(MI_BEF), "MI mean :",np.mean(MI_BEF))

ssim aft std : 0.0 ssim mean : 0.9970486760139465
ssim bef std : 0.0 ssim mean : 0.8760972619056702
psnr aft std : 0.0 psnr mean : 51.610389709472656
psnr bef std : 0.0 psnr mean : 28.216983795166016
Blur aft std : 0.0 Blur mean : 0.39639926
Blur bef std : 0.0 Blur mean : 0.43910304
MI aft std : 0.0 MI mean : 1.859090557717315
MI bef std : 0.0 MI mean : 1.2306814883200123
